In [ ]:
from langgraph.graph import StateGraph ,START , END
from typing import TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI

In [ ]:
llms = ChatGoogleGenerativeAI(model = "", api_key ="")

class EssayEvaluation(TypedDict):
    current_index: int
    Text: list[str]
    Clarity_score:list[int]
    Depth_score:list[int]
    Language_score:list[int]
    final_score:list[int]
    COT_feedback:list[str]
    DOA_feedback:list[str]
    Language_feedback:list[str]
    summarized_feedback:list[str]

In [ ]:

def evaluate_COT(state:EssayEvaluation)->EssayEvaluation:
   current_index = state['current_index'] 
   essay = state['Text'][current_index]
   prompt = f"As intellectual tutor give feedback on the following essay {essay} based on clarity of thoughts "
   prompt_2 = f"As intellectual tutor give the  score to this essay {essay} from out of ten based on clarity of thoughts"

   fb_Cot =llms.invoke(prompt)
   score_Cot= llms.invoke(prompt_2)
   state['COT_feedback'].append(fb_Cot.content)
   state['Clarity_score'].append(int(score_Cot.content))
   return state


def evaluate_DOA(state:EssayEvaluation)->EssayEvaluation:
   current_index = state['current_index'] 
   essay = state['Text'][current_index]
   prompt = f"As intellectual tutor give feedback on the following essay {essay} based on Depth of Analysis "
   prompt_2 = f"As intellectual tutor give the  score to this essay {essay} from out of ten based on Depth of Analysis"

   fb_DOA =llms.invoke(prompt)
   score_DOA= llms.invoke(prompt_2)
   state['DOA_feedback'].append(fb_DOA.content)
   state['Depth_score'].append(int(score_DOA.content))
   return state


def evaluate_language(state:EssayEvaluation)->EssayEvaluation:
   current_index = state['current_index'] 
   essay = state['Text'][current_index]
   prompt = f"As intellectual tutor give feedback on the following essay {essay} based on Language proficiency "
   prompt_2 = f"As intellectual tutor give the  score to this essay {essay} from out of ten based on Language proficiency"

   fb_Language =llms.invoke(prompt)
   score_Language= llms.invoke(prompt_2)
   state['Language_feedback'].append (fb_Language.content)
   state['Language_score'].append(int(score_Language.content))
   state['current_index'] = current_index +1 
   return state



def final_feedback_and_score(state:EssayEvaluation)->EssayEvaluation:
   prompt = f"Generate summarized_feedback on the basis of following feedbacks : clarity of thought{state['COT_feedback']} , Depth of Analysis {state['DOA_feedback']}, langauage_proficiency {state['Language_feedback']}"
   response = llms.invoke(prompt)
   state['final_score'].append((state['Clarity_score'][-1]+ state['Depth_score'][-1] + state['Language_score'][-1])/3)
   state['summarized_feedback'].append(response.content)
   return state



graph = StateGraph(EssayEvaluation)

In [ ]:
graph.add_node("COT",evaluate_COT)
graph.add_node("DOA",evaluate_DOA)
graph.add_node("Lang",evaluate_language)
graph.add_node("final",final_feedback_and_score)


graph.add_edge(START,"COT")
graph.add_edge(START,"DOA")
graph.add_edge(START,"Lang")

In [ ]:
graph.add_edge("COT", "final")
graph.add_edge("DOA", "final")
graph.add_edge("Lang", "final")
graph.add_edge("final", END)

workflow = graph.compile()

initial_state = {
    "Text": [
        "Artificial Intelligence is transforming education by helping students learn faster and more efficiently."
    ],

    "current_index": 0,

    "COT_feedback": [],
    "DOA_feedback": [],
    "Language_feedback": [],

    "Clarity_score": 0,
    "Depth_score": 0,
    "Language_score": 0,

    "final_score": 0.0,
    "summarized_feedback": ""
}

result = workflow(initial_state)


